In [ ]:
# ==============================================================================
# CELL 1: SETUP & IMPORTS
# ==============================================================================
import sys
from pathlib import Path
import pandas as pd

import plotly.io as pio
# Se estiver no JupyterLab:
pio.renderers.default = "jupyterlab"
# (alternativas: "notebook_connected" no Jupyter clássico, ou "colab" no Colab)


# --- Setup Project Path ---
# This ensures all imports from your 'src' directory work correctly.
project_root = Path.cwd().parent.parent
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# --- Import Your Analysis Toolkit ---
from hpo.analysis import (
    analyze_single_campaign, 
    aggregate_all_campaign_results, 
    analyze_across_campaigns,
    run_validation_analysis,
    analyze_holistically,
    analyze_best_per_architecture
)
from config_loader import load_campaign_config

# --- Jupyter Extensions ---
%load_ext autoreload
%autoreload 2
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 180)

In [ ]:
# ==============================================================================
# GLOBAL CONFIGURATION (The Only Cell You Edit)
# ==============================================================================
# Define the root directory for all experiment results.
RESULTS_DIR = project_root / "src" / "experiment_configs" / "results"
CONFIGS_DIR = project_root / "src" / "experiment_configs" / "hpo_campaigns"

# Define the scoring logic here ---
METRIC_WEIGHTS = {
    'val_smape_cum': 2.0,
    'val_smape_agg': 2.0
}
LOWER_IS_BETTER = {
    'val_smape_cum': True,
    'val_smape_agg': True
}
METRIC_TO_OPTIMIZE = "weighted_score"

# ==============================================================================
# DATA AGGREGATION (Run this once to load all data)
# ==============================================================================
# This function scans all your result folders and builds the master DataFrame.
master_df = aggregate_all_campaign_results(
    results_dir=RESULTS_DIR,
    metric_weights=METRIC_WEIGHTS,
    lower_is_better=LOWER_IS_BETTER
)

if master_df is not None and not master_df.empty:
    analyze_holistically(master_df, metric_to_optimize=METRIC_TO_OPTIMIZE, n_top_per_well=2)
    analyze_best_per_architecture(master_df, metric_to_optimize=METRIC_TO_OPTIMIZE)
else:
    print("Master DataFrame is empty. Run some campaigns first.")

In [ ]:
# ==============================================================================
# ANALYSIS & VISUALIZATION
# ==============================================================================
# This script generates various plots to analyze architecture performance
# and per-well hyperparameter importance.

from hpo.analysis import (
    plot_performance_by_architecture,
    plot_champions_per_well,
    plot_hyperparameter_importance_per_well,
)

def generate_plots(master_df, metric_to_optimize):
    """
    Generate performance and hyperparameter importance plots
    if the DataFrame is non-empty.
    """
    if master_df is None or master_df.empty:
        print("Master DataFrame is empty. Cannot generate plots.")
        return

    # ------------------------------------------------------
    # 1. Architecture Performance Analysis
    # ------------------------------------------------------
    # Compare architectures in terms of performance consistency
    plot_performance_by_architecture(master_df, metric=metric_to_optimize)

    # Visual breakdown of the "Champions League" table
    plot_champions_per_well(master_df, metric=metric_to_optimize)

    # ------------------------------------------------------
    # 2. Per-Well Hyperparameter Deep Dive
    # ------------------------------------------------------
    # Identify which hyperparameters are most influential for each well
    plot_hyperparameter_importance_per_well(
        master_df, metric_col=metric_to_optimize
    )

generate_plots(master_df, METRIC_TO_OPTIMIZE)